# Notebook 07 — Market Metrics

**Input :** `data/unified_dataset.parquet` — dataset filtré (168 lignes, step 05)

**Output :** `data/unified_with_metrics.parquet` — dataset enrichi (168 lignes, 18 colonnes)

**Métriques calculées via `src/metrics.py` :**

| Colonne | Description |
|---|---|
| `Avg_Volume_MC` | Volume moyen traité (Volume MC) |
| `Liquidity_Proxy` | Proxy liquidité = Volume moyen × Prix moyen |
| `Avg_Spread` | Spread Bid-Ask moyen |
| `Min/Max/Std_Spread` | Statistiques du spread |
| `Avg/Std_Spread_Pct` | Spread relatif en % du Bid |
| `Trading_Coverage_Pct` | % sessions avec données de volume |
| `Volatility_Proxy_Pct` | Coefficient de variation des prix |

In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# ROOT resolution: works regardless of where the Jupyter kernel starts.
# VS Code / Kiro kernels start from the project root (DSS_CMR/).
# Classic Jupyter starts from the notebooks/ folder.
# We detect ROOT by searching upward for the 'src' and 'data' directories.
def _find_root(start: Path) -> Path:
    for candidate in [start, start.parent, start.parent.parent]:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate.resolve()
    raise RuntimeError(f'Cannot locate project root from {start}')

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.validation import load_unified_dataset, save_unified_dataset
from src.metrics import compute_all_metrics, get_metrics_summary

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print(f'ROOT   : {ROOT}')
print(f'cwd    : {Path.cwd()}')
print(f'Input  : {ROOT / "data" / "unified_dataset.parquet"}')
print(f'Output : {ROOT / "data" / "unified_with_metrics.parquet"}')

ROOT   : /home/yass/Desktop/DSS_CMR
cwd    : /home/yass/Desktop/DSS_CMR/notebooks
Input  : /home/yass/Desktop/DSS_CMR/data/unified_dataset.parquet
Output : /home/yass/Desktop/DSS_CMR/data/unified_with_metrics.parquet


## Step 1 — Chargement depuis Parquet

In [3]:
unified, _ = load_unified_dataset(str(ROOT / 'data' / 'unified_dataset.parquet'))

print(f'Shape      : {unified.shape}')
print(f'Sociétés   : {unified["CODE_ISIN"].nunique()}')
print(f'Sessions   : {unified["Date"].nunique()}')
print(f'Période    : {unified["Date"].min().date()} → {unified["Date"].max().date()}')
print(f'Colonnes   : {list(unified.columns)}')
print()
print(unified[['CODE_ISIN','Company']].drop_duplicates().to_string(index=False))

Shape      : (168, 8)
Sociétés   : 6
Sessions   : 28
Période    : 2018-12-31 → 2024-01-19
Colonnes   : ['Date', 'CODE_ISIN', 'Company', 'Ask', 'Bid', 'Cours', 'Quantité MC', 'Volume MC']

   CODE_ISIN             Company
MA0000010936  ALUMINIUM DU MAROC
MA0000010944                AGMA
MA0000010951        AFRIQUIA GAZ
MA0000011819           ALLIANCES
MA0000012114 AFRIC INDUSTRIES SA
MA0000012296                AFMA
MA0000012114    AFRIC INDUSTRIES


## Step 2 — Couverture des données par société

In [4]:
key_cols = ['Cours', 'Bid', 'Ask', 'Volume MC', 'Quantité MC']
isin_to_name = unified.drop_duplicates('CODE_ISIN').set_index('CODE_ISIN')['Company']

coverage = unified.groupby('CODE_ISIN')[key_cols].apply(
    lambda df: (df.notna().sum() / len(df) * 100).round(1)
)
coverage.index = coverage.index.map(isin_to_name)

print('Couverture (% sessions non-nulles) :')
print(coverage.to_string())

Couverture (% sessions non-nulles) :
                      Cours     Bid     Ask  Volume MC  Quantité MC
CODE_ISIN                                                          
ALUMINIUM DU MAROC  50.0000 50.0000 50.0000    17.9000      17.9000
AGMA                50.0000 25.0000 50.0000     3.6000       3.6000
AFRIQUIA GAZ        50.0000 50.0000 50.0000    17.9000      17.9000
ALLIANCES           50.0000 50.0000 50.0000    50.0000      50.0000
AFRIC INDUSTRIES SA 50.0000 50.0000 50.0000     7.1000       7.1000
AFMA                50.0000 50.0000 50.0000    17.9000      17.9000


## Step 3 — Calcul des métriques via src/metrics.py

In [5]:
unified_with_metrics, report = compute_all_metrics(unified, verbose=True)

print(f'\nShape avant  : {unified.shape}')
print(f'Shape après  : {unified_with_metrics.shape}')
print(f'Calculées    : {report["metrics_computed"]}')

if report['warnings']:
    print('\nAvertissements :')
    for w in report['warnings']:
        print(f'  {w}')

Computing market metrics...

1. Average Volume (Volume MC)
   ✓ 6 companies

2. Liquidity Proxy (Volume × Price)
   ✓ 6 companies

3. Bid-Ask Spread Statistics
   ✓ 6 companies

4. Trading Coverage (%)
   ✓ Coverage computed

5. Volatility Proxy (Coefficient of Variation)
   ✓ Volatility proxy computed

✓ Metrics computation complete
  Computed: 10 metrics
  Skipped: 0 metrics

Shape avant  : (168, 8)
Shape après  : (168, 18)
Calculées    : ['Avg_Volume_MC', 'Liquidity_Proxy', 'Avg_Spread', 'Min_Spread', 'Max_Spread', 'Std_Spread', 'Avg_Spread_Pct', 'Std_Spread_Pct', 'Trading_Coverage_Pct', 'Volatility_Proxy_Pct']


## Step 4 — Résumé par société

In [6]:
summary = get_metrics_summary(unified_with_metrics)
print('Métriques agrégées par société :')
print(summary.to_string())

Métriques agrégées par société :
                          Company  Avg_Volume_MC    Liquidity_Proxy  Avg_Spread  Avg_Spread_Pct  Trading_Coverage_Pct  Volatility_Proxy_Pct
CODE_ISIN                                                                                                                                  
MA0000010936   ALUMINIUM DU MAROC    34,638.8000    57,651,334.2000     28.7857          2.1849               17.8571                2.6932
MA0000010944                 AGMA     3,040.0000     9,292,411.4286    431.8571          6.9816                3.5714                0.6552
MA0000010951         AFRIQUIA GAZ   341,070.4000 1,058,000,380.8000    192.6429          4.8580               17.8571                4.4597
MA0000011819            ALLIANCES   456,907.7686    36,270,317.7573      2.3464          1.7229               50.0000                4.0877
MA0000012114  AFRIC INDUSTRIES SA     1,630.5000       446,663.8286      8.8286          2.7543                7.1429          

## Step 5 — Qualité des colonnes ajoutées

In [7]:
new_cols = [c for c in unified_with_metrics.columns if c not in unified.columns]

print(f'{"Colonne":30s}  {"Non-null":>8}  {"Couverture":>11}')
print('-' * 55)
for col in new_cols:
    n   = unified_with_metrics[col].notna().sum()
    pct = n / len(unified_with_metrics) * 100
    print(f'{col:30s}  {n:>8d}  {pct:>10.1f}%')

Colonne                         Non-null   Couverture
-------------------------------------------------------
Avg_Volume_MC                        168       100.0%
Liquidity_Proxy                      168       100.0%
Avg_Spread                           168       100.0%
Min_Spread                           168       100.0%
Max_Spread                           168       100.0%
Std_Spread                           168       100.0%
Avg_Spread_Pct                       168       100.0%
Std_Spread_Pct                       168       100.0%
Trading_Coverage_Pct                 168       100.0%
Volatility_Proxy_Pct                 168       100.0%


## Step 6 — Statistiques descriptives

In [8]:
for col in new_cols:
    s = unified_with_metrics[col].dropna()
    if len(s) == 0:
        continue
    print(f'{col}')
    print(f'  n={len(s)}  min={s.min():,.2f}  médiane={s.median():,.2f}  moyenne={s.mean():,.2f}  max={s.max():,.2f}')
    print()

Avg_Volume_MC
  n=168  min=1,630.50  médiane=67,868.70  moyenne=156,397.68  max=456,907.77

Liquidity_Proxy
  n=168  min=446,663.83  médiane=46,960,825.98  moyenne=210,018,246.74  max=1,058,000,380.80

Avg_Spread
  n=168  min=2.35  médiane=42.39  moyenne=120.08  max=431.86

Min_Spread
  n=168  min=0.50  médiane=20.50  moyenne=42.90  max=114.00

Max_Spread
  n=168  min=3.80  médiane=66.50  moyenne=216.04  max=790.00

Std_Spread
  n=168  min=0.96  médiane=15.73  moyenne=49.88  max=202.54

Avg_Spread_Pct
  n=168  min=1.72  médiane=3.80  moyenne=3.89  max=6.98

Std_Spread_Pct
  n=168  min=0.70  médiane=1.63  moyenne=1.66  max=3.33

Trading_Coverage_Pct
  n=168  min=3.57  médiane=17.86  moyenne=19.05  max=50.00

Volatility_Proxy_Pct
  n=168  min=0.66  médiane=2.84  moyenne=2.93  max=4.46



## Step 7 — Aperçu du dataset enrichi

In [9]:
print(f'Dataset enrichi : {unified_with_metrics.shape}')
print()
print(unified_with_metrics.head(10).to_string(index=False))

Dataset enrichi : (168, 18)

      Date    CODE_ISIN             Company  Ask  Bid      Cours  Quantité MC      Volume MC  Avg_Volume_MC    Liquidity_Proxy  Avg_Spread  Min_Spread  Max_Spread  Std_Spread  Avg_Spread_Pct  Std_Spread_Pct  Trading_Coverage_Pct  Volatility_Proxy_Pct
2018-12-31 MA0000010936  ALUMINIUM DU MAROC  NaN  NaN 1,565.0000          NaN            NaN    34,638.8000    57,651,334.2000     28.7857     16.0000     48.0000     12.3798          2.1849          0.9395               17.8571                2.6932
2018-12-31 MA0000010944                AGMA  NaN  NaN 3,079.0000          NaN            NaN     3,040.0000     9,292,411.4286    431.8571     99.0000    790.0000    202.5376          6.9816          3.3303                3.5714                0.6552
2018-12-31 MA0000010951        AFRIQUIA GAZ  NaN  NaN 3,000.0000          NaN            NaN   341,070.4000 1,058,000,380.8000    192.6429    114.0000    348.0000     58.8670          4.8580          1.5737            

## Step 8 — Sauvegarde Parquet

In [10]:
out_path = str(ROOT / 'data' / 'unified_with_metrics.parquet')
rep = save_unified_dataset(unified_with_metrics, out_path, compression='snappy')

print(f'✓ Sauvegardé  → {out_path}')
print(f'  rows={rep["rows"]}  cols={rep["columns"]}  size={rep["file_size_mb"]:.3f} MB')

# vérification round-trip
check, _ = load_unified_dataset(out_path)
assert check.shape == unified_with_metrics.shape, f'Shape mismatch: {check.shape}'
print(f'  Round-trip ✓  {check.shape}')

✓ Sauvegardé  → /home/yass/Desktop/DSS_CMR/data/unified_with_metrics.parquet
  rows=168  cols=18  size=0.013 MB
  Round-trip ✓  (168, 18)


## Résumé

In [11]:
print('=' * 65)
print('NOTEBOOK 07 — MÉTRIQUES DE MARCHÉ — RÉSUMÉ')
print('=' * 65)
print(f'  Input  : {unified.shape}  →  unified_dataset.parquet')
print(f'  Output : {unified_with_metrics.shape}  →  unified_with_metrics.parquet')
print()
print('  Métriques calculées :')
for m in report['metrics_computed']:
    print(f'    ✓  {m}')
if report['metrics_skipped']:
    print('  Métriques reportées (données insuffisantes) :')
    for m in report['metrics_skipped']:
        print(f'    ⚠  {m}')
print()
print('  Prochain notebook : 08 — Filtrage Dynamique')
print('=' * 65)

NOTEBOOK 07 — MÉTRIQUES DE MARCHÉ — RÉSUMÉ
  Input  : (168, 8)  →  unified_dataset.parquet
  Output : (168, 18)  →  unified_with_metrics.parquet

  Métriques calculées :
    ✓  Avg_Volume_MC
    ✓  Liquidity_Proxy
    ✓  Avg_Spread
    ✓  Min_Spread
    ✓  Max_Spread
    ✓  Std_Spread
    ✓  Avg_Spread_Pct
    ✓  Std_Spread_Pct
    ✓  Trading_Coverage_Pct
    ✓  Volatility_Proxy_Pct

  Prochain notebook : 08 — Filtrage Dynamique
